### Inference script for the Segformer-based artifact detector.

In [1]:
import logging
from pathlib import Path
import sys
import os

import cv2
import numpy as np
import pandas as pd  # Added pandas for CSV handling
import torch
import torch.nn as nn
from torchvision import transforms
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

from google.colab import drive

In [2]:
# configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger: logging.Logger = logging.getLogger(__name__)

np.set_printoptions(threshold=sys.maxsize)

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
def initialize_segformer(
    model_name_or_path: str | Path, out_channels: int = 1
) -> tuple[SegformerImageProcessor, SegformerForSemanticSegmentation]:
    """Loads and modifies a pre-trained Segformer model for artifact detection."""

    logger.info(f"Loading Segformer model from: {model_name_or_path}")

    # load a pretrained Segformer model
    preprocessor = SegformerImageProcessor.from_pretrained(model_name_or_path)
    model = SegformerForSemanticSegmentation.from_pretrained(model_name_or_path)

    # change the number of output channels
    in_channels = model.decode_head.classifier.in_channels
    model.decode_head.classifier = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    return preprocessor, model

In [5]:
def process_images(
    csv_path: str,
    output_heatmap_dir: str,
    output_mask_dir: str,
    output_mask_txt_dir: str,
    model_weights_path: str,
    device: torch.device,
) -> None:
    """Runs artifact detection inference on a filtered list of images from a CSV."""

    # Create directories using os.makedirs
    os.makedirs(output_heatmap_dir, exist_ok=True)
    os.makedirs(output_mask_dir, exist_ok=True)
    os.makedirs(output_mask_txt_dir, exist_ok=True)

    seg_preprocessor, artifact_detector = initialize_segformer("nvidia/mit-b5", out_channels=1)

    logger.info(f"Loading weights from {model_weights_path}")
    state_dict = torch.load(model_weights_path, map_location=device, weights_only=True)
    artifact_detector.load_state_dict(state_dict)

    artifact_detector.to(device)
    artifact_detector.eval()

    logger.info(f"Loading and filtering CSV dataset from: {csv_path}")
    df = pd.read_csv(csv_path)

    # Filter for test split and AI label
    filtered_df = df[(df['split'] == 'test') & (df['label'] == 'ai')]

    # Extract file paths as a list of strings
    image_paths = filtered_df['file_path'].tolist()

    if not image_paths:
        logger.warning(f"No valid images found after filtering {csv_path}")
        return

    logger.info(f"Starting inference on {len(image_paths)} filtered images...")

    for image_path in tqdm(image_paths, desc="Processing Images"):
        # Extract the filename without the extension
        image_name: str = os.path.splitext(os.path.basename(image_path))[0]

        raw_image: np.ndarray = cv2.imread(image_path)
        if raw_image is None:
            logger.error(f"Failed to read image: {image_path}")
            continue

        resized_image: np.ndarray = cv2.resize(raw_image, (512, 512))
        rgb_image: np.ndarray = cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB)
        visualizable_image: np.ndarray = resized_image.copy()

        with torch.no_grad():
            tensor_image: torch.Tensor = transforms.ToTensor()(rgb_image).to(device)
            processed_inputs = seg_preprocessor(
                tensor_image, return_tensors="pt", do_rescale=False
            )
            pixel_values: torch.Tensor = processed_inputs["pixel_values"].to(device)

            model_output = artifact_detector(pixel_values)

            upsampled_logits: torch.Tensor = nn.functional.interpolate(
                model_output.logits,
                size=pixel_values.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            normalized_predictions: torch.Tensor = torch.sigmoid(upsampled_logits)

        # reformat the tensor output back into a valid cv2 image format
        mask_array: np.ndarray = (
            normalized_predictions[0].detach().cpu().numpy().transpose(-2, -1, -3) * 255
        ).astype(np.uint8)

        # save the raw numpy array as txt file using os.path.join
        mask_txt_save_path = os.path.join(output_mask_txt_dir, f"mask_{image_name}.txt")
        with open(mask_txt_save_path, "w") as f:
            f.write(str(mask_array))

        heatmap_color: np.ndarray = cv2.applyColorMap(mask_array, cv2.COLORMAP_JET)
        heatmap_alpha: float = 0.6 # the transparency of the heatmap
        blended_heatmap: np.ndarray = cv2.addWeighted(
            heatmap_color, heatmap_alpha, visualizable_image, 1 - heatmap_alpha, 0
        )

        heatmap_save_path = os.path.join(output_heatmap_dir, f"result_{image_name}.png")
        cv2.imwrite(heatmap_save_path, blended_heatmap)

        _, binary_mask = cv2.threshold(mask_array, 127, 255, cv2.THRESH_BINARY)

        # Dilation: Expands the white areas slightly so the downstream inpainting
        # model has a clean edge buffer to blend the synthesized textures with.
        dilation_kernel: np.ndarray = np.ones((9, 9), np.uint8)
        dilated_mask: np.ndarray = cv2.dilate(binary_mask, dilation_kernel, iterations=1)

        mask_save_path = os.path.join(output_mask_dir, f"mask_{image_name}.png")
        cv2.imwrite(mask_save_path, dilated_mask)

    logger.info("Done!")

In [6]:
# configure paths
adinf_dir= '/content/drive/MyDrive/Colab Notebooks/deepfake-it-till-you-make-it/src/model/discriminator/artifact_detector'

cfg_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg_weights = adinf_dir + "/checkpoints/ad_richhf_baseline_model.bin"
cfg_csv_path = '/'.join(adinf_dir.split('/')[:-6]) + '/TrainingData/dataset_inventory.csv'
cfg_heatmap_dir = '/'.join(adinf_dir.split('/')[:-6]) + "/output/heatmap/"
cfg_mask_dir = '/'.join(adinf_dir.split('/')[:-6]) + "/output/mask/"
cfg_mask_txt_dir = '/'.join(adinf_dir.split('/')[:-6]) + "/output/mask_txt/"

process_images(
    csv_path=cfg_csv_path,
    output_heatmap_dir=cfg_heatmap_dir,
    output_mask_dir=cfg_mask_dir,
    output_mask_txt_dir=cfg_mask_txt_dir,
    model_weights_path=cfg_weights,
    device=cfg_device,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1156 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b5
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta